# 01 — Inspect the raw dataset

Converted from `scripts/01_inspect.py` (Derrick, data prep).

First look at the full **FreshRetailNet-50K** dataset before cutting the working subset: shape, columns, types, a sample row, distinct counts, date range, and how many series are complete.

> Run this notebook from the `Data Prep - Derrick` folder. The first cell downloads the full dataset from the Hugging Face Hub (large, cached after the first run).

### Where this fits — CRISP-DM Phase 2: Data Understanding

This notebook is the **Data Understanding** phase of CRISP-DM (Week 5): *collect the
initial data, describe it, explore it, and verify its quality* before we build
anything.

**The dataset.** FreshRetailNet-50K (Dingdong-Inc, Hugging Face) is a fresh-grocery
retail dataset. The published size is **4.5 million training rows + 350,000
evaluation rows**. The "50K" in the name refers to about **50,000 store-product
series** (4,500,000 / 90 days = 50,000): each series is one product in one store
followed over 90 daily records. Every row carries scaled daily sales, 24-hour sales
and stock-status arrays, promotion/holiday flags, a discount ratio, stockout hours,
and weather.

**What the cells below do, in CRISP-DM terms:**

| CRISP-DM task (Phase 2) | Cell(s) in this notebook |
| --- | --- |
| Collect initial data | `load_dataset(...)` |
| Describe data | row / column counts, column types, one sample row |
| Explore data | distinct counts, date range, stores & series per city |
| Verify data quality | days-per-series, count of complete 90-day series |

**Levels of measurement (Week 6).** Classifying each variable tells us which
operations are valid — e.g. an id is a label, so averaging it is meaningless:

| Variable | Measurement level |
| --- | --- |
| `city_id`, `store_id`, `product_id`, `management_group_id`, the category ids | Nominal (categorical codes) |
| `holiday_flag`, `activity_flag` | Nominal / binary |
| `dt` | Interval (calendar date) |
| `avg_temperature` | Interval (0 is arbitrary) |
| `sale_amount`, `hours_sale`, `discount`, `stock_hour6_22_cnt`, `precpt`, `avg_humidity`, `avg_wind_level` | Ratio (true zero) |

This is why Notebook 02 stores the id columns as `category`, not numbers.

In [1]:
# MIS 480 group project - look at the raw dataset before cutting our subset
# Derrick, data prep
import pandas as pd
from datasets import load_dataset

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [3]:
ds = load_dataset("Dingdong-Inc/FreshRetailNet-50K", split="train")

README.md:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

C:\Users\GPU\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\GPU\.cache\huggingface\hub\datasets--Dingdong-Inc--FreshRetailNet-50K. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

data/eval.parquet:   0%|          | 0.00/8.44M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4500000 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/350000 [00:00<?, ? examples/s]

In [4]:
print(f"rows: {ds.num_rows}")
print(f"columns: {ds.column_names}")

rows: 4500000
columns: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level']


In [5]:
print("\ncolumn types:")
for name, feat in ds.features.items():
    print(f"  {name}: {feat}")


column types:
  city_id: Value('int64')
  store_id: Value('int64')
  management_group_id: Value('int64')
  first_category_id: Value('int64')
  second_category_id: Value('int64')
  third_category_id: Value('int64')
  product_id: Value('int64')
  dt: Value('string')
  sale_amount: Value('float64')
  hours_sale: List(Value('float64'))
  stock_hour6_22_cnt: Value('int32')
  hours_stock_status: List(Value('int64'))
  discount: Value('float64')
  holiday_flag: Value('int32')
  activity_flag: Value('int32')
  precpt: Value('float64')
  avg_temperature: Value('float64')
  avg_humidity: Value('float64')
  avg_wind_level: Value('float64')


In [6]:
print("\nsample row:")
for k, v in ds[0].items():
    show = v if not isinstance(v, list) else f"list(len={len(v)}): {v[:6]}..."
    print(f"  {k}: {show}")


sample row:
  city_id: 0
  store_id: 0
  management_group_id: 0
  first_category_id: 5
  second_category_id: 6
  third_category_id: 65
  product_id: 38
  dt: 2024-03-28
  sale_amount: 0.1
  hours_sale: list(len=24): [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]...
  stock_hour6_22_cnt: 0
  hours_stock_status: list(len=24): [0, 0, 0, 0, 0, 0]...
  discount: 1.0
  holiday_flag: 0
  activity_flag: 0
  precpt: 1.6999
  avg_temperature: 15.48
  avg_humidity: 73.54
  avg_wind_level: 1.97


In [7]:
print("\ndistinct counts:")
for col in ["city_id", "store_id", "product_id", "dt"]:
    print(f"  {col}: {len(ds.unique(col))}")


distinct counts:
  city_id: 18
  store_id: 898
  product_id: 865
  dt: 90


In [8]:
idx = ds.select_columns(["city_id", "store_id", "product_id", "dt"]).to_pandas()
idx["dt"] = pd.to_datetime(idx["dt"])

In [9]:
print(f"\ndate range: {idx['dt'].min().date()} to {idx['dt'].max().date()} ({idx['dt'].nunique()} days)")


date range: 2024-03-28 to 2024-06-25 (90 days)


In [10]:
print("\nstores per city (top 10):")
print(idx.groupby("city_id")["store_id"].nunique().sort_values(ascending=False).head(10).to_string())


stores per city (top 10):
city_id
0     290
12    107
3     101
13     90
16     89
6      43
11     38
4      34
7      20
15     17


In [11]:
print("\nseries per city (top 10):")
print(idx.groupby("city_id").apply(lambda g: g.groupby(["store_id","product_id"]).ngroups).sort_values(ascending=False).head(10).to_string())


series per city (top 10):
city_id
0     25811
12     5117
16     4528
3      2648
6      2384
13     2336
11     1870
4      1141
15      803
5       677


In [12]:
days = idx.groupby(["store_id","product_id"])["dt"].nunique()
print(f"\ndays per series: min {days.min()}, max {days.max()}")
print(f"complete (90-day) series: {(days == 90).sum()} of {len(days)}")


days per series: min 90, max 90
complete (90-day) series: 50000 of 50000


---
**Pipeline:** **01 inspect** → 02 subset → 03 EDA → 04 modeling → 05 classification.

**Phase 2 outcome.** The data is well-formed (0 missing, complete 90-day series) and we understand the variables and their measurement levels. This hands off to **Phase 3 (notebook 02)**, where we select the working subset and engineer features.